In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['JAX_PLATFORM_NAME'] = 'cpu'  # Avoid Metal issues on M1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import jax
from pprint import pprint

from tipbox.generate import generate
from tipbox import registry

# High-Level API

### 0. Types of Tipping Supported

In [ ]:
# You can list the available tipping types with:
# This will have to be passed to the generate function as the `tipping_type` argument
registry.list_systems()

### 1. Bifurcation-induced

In [ ]:
# Generate
full, part, info = generate(
    key=jax.random.PRNGKey(123),
    steps_before_transition=1,
    num_timesteps=500,
    sigma_tilde=0.01,
    tipping_type='b_harvesting',
    null_simulation=False,
    param_end_multiplier=2.0
)

In [ ]:
pprint(info)

In [ ]:
# Visualize the trajectory
full_time = np.asarray(full.time)
full_traj = np.asarray(full.trajectory)
part_traj = np.asarray(part.trajectory)

transition_idx = int(full.metadata.get("transition_index", full_traj.shape[0]))
window_len = part_traj.shape[0]
window_start_idx = max(0, transition_idx - window_len - full.metadata.get("steps_before_transition", 0))
window_end_idx = min(window_start_idx + window_len, full_traj.shape[0])

label = str(full.metadata.get("label", "unknown")).upper()

def extraction_span(ax):
    """Highlight the extraction window on time-series plots."""
    ax.axvspan(full_time[window_start_idx],
               full_time[window_end_idx - 1],
               color="C2", alpha=0.20, label="Extraction window")

# Trajectory
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(full_time, full_traj[:, 0], color="C0", label=f"{full.metadata.get('label', 'unknown').capitalize()}")
axes[0].axvline(transition_idx, linestyle="--", color="black", label="Transition")
extraction_span(axes[0])
axes[0].set_xlabel("Time")
axes[0].set_ylabel("x")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

axes[1].plot(full_time, full_traj[:, 1], color="C0", label=f"{full.metadata.get('label', 'unknown').capitalize()}")
axes[1].axvline(transition_idx, linestyle="--", color="black", label="Transition")
extraction_span(axes[1])
axes[1].set_xlabel("Time")
axes[1].set_ylabel("y")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

#  Phase space
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(full_traj[:, 0], full_traj[:, 1], color="lightgray", linewidth=1.5, label="Full trajectory")
axes[0].plot(full_traj[window_start_idx:window_end_idx, 0],
             full_traj[window_start_idx:window_end_idx, 1],
             color="C2", linewidth=2.0, label="Extraction window")

axes[0].scatter(full_traj[0, 0], full_traj[0, 1], marker="o", color="C1", edgecolor="k", label="Start", zorder=5)
axes[0].scatter(full_traj[-1, 0], full_traj[-1, 1], marker="x", color="C1", label="End", zorder=6)

axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

# Rolling variance
window = 50  # number of points for the rolling variance window
rolling_var = [
    np.var(full_traj[max(0, i - window):i, 0])
    for i in range(window, full_traj.shape[0])
]
var_time = full_time[window:window + len(rolling_var)]

axes[1].plot(var_time, rolling_var, color="C0", linewidth=1.5, label="Rolling variance")
axes[1].axvline(transition_idx, linestyle="--", color="black", label="Transition")
extraction_span(axes[1])
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

### 2. Rate-induced

In [ ]:
import jax
from tipbox.generate import generate

full, part, info = generate(
    key=jax.random.PRNGKey(123),
    steps_before_transition=1,
    num_timesteps=500,
    sigma_tilde=0.01,
    tipping_type='r_compost_bomb',
    null_simulation=True,
)

In [ ]:
pprint(info)

In [ ]:
# Visualize the trajectory
full_time = np.asarray(full.time)
full_traj = np.asarray(full.trajectory)
part_traj = np.asarray(part.trajectory)

transition_idx = int(full.metadata.get("transition_index", full_traj.shape[0]))
transition_time = float(full_time[min(transition_idx, full_time.size - 1)])
window_len = part_traj.shape[0]
window_start_idx = max(0, transition_idx - window_len - full.metadata.get("steps_before_transition", 0))
window_end_idx = min(window_start_idx + window_len, full_traj.shape[0])

def add_extraction_span(ax):
    ax.axvspan(
        float(full_time[window_start_idx]), 
        float(full_time[window_end_idx - 1]),
        color="C2", alpha=0.20, label="Extraction window"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trajectory
axes[0].plot(full_time, full_traj[:, 1], color="C0", label=f"{full.metadata.get('rate_system', 'unknown').capitalize()}")
axes[0].axvline(transition_time, linestyle="--", color="black", label="Transition")
add_extraction_span(axes[0])
axes[0].set_xlabel("Time")
axes[0].set_ylabel("x")
# axes[0].set_ylim(full_traj[0,0], info['system_config']['tip_threshold'])
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

# Rolling variance
window = 50
rolling_var = np.array([
    np.var(full_traj[max(0, i - window):i, 1])
    for i in range(window, full_traj.shape[0])
])
var_time = full_time[window:window + len(rolling_var)]

axes[1].plot(var_time, rolling_var, color="C0", linewidth=1.5, label="Rolling variance")
axes[1].axvline(transition_time, linestyle="--", color="black", label="Transition")
add_extraction_span(axes[1])
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

### 3. Box Models

In [ ]:
# Generate
full, part, info = generate(
    key=jax.random.PRNGKey(123),
    steps_before_transition=1,
    num_timesteps=500,
    sigma_tilde=1.0,
    tipping_type='b_seirx_tc',
    null_simulation=False,
    param_end_multiplier=1.1
)

In [ ]:
pprint(info)

In [ ]:
# Visualize the trajectory
full_time = np.asarray(full.time)
full_traj = np.asarray(full.trajectory)
part_traj = np.asarray(part.trajectory)

transition_idx = int(full.metadata.get("transition_index", full_traj.shape[0]))
window_len = part_traj.shape[0]
window_start_idx = max(0, transition_idx - window_len - full.metadata.get("steps_before_transition", 0))
window_end_idx = min(window_start_idx + window_len, full_traj.shape[0])

label = str(full.metadata.get("label", "unknown")).upper()

def extraction_span(ax):
    """Highlight the extraction window on time-series plots."""
    ax.axvspan(full_time[window_start_idx],
               full_time[window_end_idx - 1],
               color="C2", alpha=0.20, label="Extraction window")

# Trajectory
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
param_idx = 0

axes[0].plot(full_time, full_traj[:, param_idx], color="C0", label=f"{full.metadata.get('label', 'unknown').capitalize()}")
axes[0].axvline(transition_idx, linestyle="--", color="black", label="Transition")
extraction_span(axes[0])
axes[0].set_xlabel("Time")
axes[0].set_ylabel("x")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

# Rolling variance
window = 50  # number of points for the rolling variance window
rolling_var = [
    np.var(full_traj[max(0, i - window):i, param_idx])
    for i in range(window, full_traj.shape[0])
]
var_time = full_time[window:window + len(rolling_var)]

axes[1].plot(var_time, rolling_var, color="C0", linewidth=1.5, label="Rolling variance")
axes[1].axvline(transition_idx, linestyle="--", color="black", label="Transition")
extraction_span(axes[1])
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()